In [1]:
# analysis.ipynb
# Run each cell in order

# Cell 1: Imports
import sys
sys.path.insert(0, "backend")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

plt.style.use("dark_background")
RESULTS = Path("experiments/results")

print("✓ Imports OK")
print(f"✓ Results dir: {RESULTS.exists()}")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Cell 2: Load All Results
df      = pd.read_csv(RESULTS / "results.csv")
summary = pd.read_csv(RESULTS / "summary.csv")
cf      = pd.read_csv(RESULTS / "counterfactual.csv")
sens    = pd.read_csv(RESULTS / "sensitivity.csv")
sweep   = pd.read_csv(RESULTS / "param_sweep_1000.csv")

try:
    carla = pd.read_csv(RESULTS / "carla_results.csv")
    print(f"✓ CARLA results: {len(carla)} rows")
except:
    carla = None
    print("  CARLA results not found — skipping")

print(f"✓ Ablation:      {len(df)} trials")
print(f"✓ Summary:       {len(summary)} rows")
print(f" Counterfactual:{len(cf)} rows")
print(f" Sensitivity:   {len(sens)} rows")
print(f" Param sweep:   {len(sweep)} configs")

In [ ]:
# Cell 3: Headline Results
print("=" * 55)
print("  HEADLINE RESULTS")
print("=" * 55)

policies  = ["greedy", "hesitation", "fixed_delay",
             "random_delay", "risk_threshold", "ttc_only"]
scenarios = df["scenario"].unique()

for scenario in scenarios:
    print(f"\n  {scenario}:")
    baseline = summary[
        (summary.scenario == scenario) &
        (summary.policy   == "greedy")
    ]["hqm_mean"].values[0]

    for policy in policies:
        row = summary[
            (summary.scenario == scenario) &
            (summary.policy   == policy)
        ]
        if len(row) == 0:
            continue
        hqm   = row["hqm_mean"].values[0]
        std   = row["hqm_std"].values[0]
        delta = hqm - baseline
        win   = "" if delta > 0 else "✗"
        print(f"    {win} {policy:<18} "
              f"HQM={hqm:.4f} ±{std:.4f}  "
              f"Δ={delta:+.4f}")

In [ ]:
# Cell 4: Six-Policy HQM Bar Chart
fig, ax = plt.subplots(figsize=(14, 5),
                        facecolor="#0f172a")
ax.set_facecolor("#020617")

policy_colors = {
    "greedy":          "#ef4444",
    "hesitation":      "#3b82f6",
    "fixed_delay":     "#22c55e",
    "random_delay":    "#eab308",
    "risk_threshold":  "#a855f7",
    "ttc_only":        "#f97316",
}
policy_labels = {
    "greedy":          "Greedy",
    "hesitation":      "Hesitation",
    "fixed_delay":     "Fixed Delay",
    "random_delay":    "Random Delay",
    "risk_threshold":  "Risk Threshold",
    "ttc_only":        "TTC-Only",
}

scenarios = list(df["scenario"].unique())
x = np.arange(len(scenarios))
w = 0.13

for i, policy in enumerate(policies):
    means, stds = [], []
    for scenario in scenarios:
        row = summary[
            (summary.scenario == scenario) &
            (summary.policy   == policy)
        ]
        means.append(row["hqm_mean"].values[0]
                     if len(row) > 0 else 0.0)
        stds.append(row["hqm_std"].values[0]
                    if len(row) > 0 else 0.0)

    ax.bar(x + i*w, means, w,
           yerr=stds, capsize=3,
           label=policy_labels[policy],
           color=policy_colors[policy],
           alpha=0.85, ecolor="#475569",
           edgecolor="#020617")

ax.axhline(y=0.60, color="#475569",
            linestyle="--", linewidth=1,
            label="Greedy baseline")
ax.set_xticks(x + w * 2.5)
ax.set_xticklabels(
    [s.replace("_", "\n") for s in scenarios],
    fontsize=9, color="#94a3b8")
ax.set_ylabel("Mean HQM", color="#64748b")
ax.set_title("Six-Policy Comparison",
              color="#e2e8f0", fontsize=12)
ax.set_ylim(0, 0.9)
ax.tick_params(colors="#475569")
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.legend(fontsize=8, facecolor="#0f172a",
           labelcolor="#94a3b8",
           edgecolor="#1e293b",
           loc="upper right")

plt.tight_layout()
plt.savefig("paper_figures/six_policy_bar.png",
            dpi=150, facecolor="#0f172a")
plt.show()
print("✓ Saved paper_figures/six_policy_bar.png")

In [ ]:
# Cell 5: HQM Component Breakdown
fig, axes = plt.subplots(1, 3, figsize=(14, 4),
                          facecolor="#0f172a")
fig.suptitle("HQM Component Breakdown — Hesitation Policy",
              color="#e2e8f0", fontsize=11)

comp_colors = ["#818cf8", "#22c55e", "#eab308", "#f97316"]
comp_labels = ["S (Safety)", "E (Efficiency)",
               "B (Stability)", "R (Resolution)"]

hes = summary[summary.policy == "hesitation"]

for idx, scenario in enumerate(scenarios):
    ax  = axes[idx]
    ax.set_facecolor("#020617")
    row = hes[hes.scenario == scenario]

    vals = [
        row["S_mean"].values[0],
        row["E_mean"].values[0],
        row["B_mean"].values[0],
        row["R_mean"].values[0],
    ]

    bars = ax.bar(["S", "E", "B", "R"], vals,
                   color=comp_colors,
                   edgecolor="#020617")
    ax.set_ylim(0, 1.2)
    ax.set_title(scenario.replace("_", "\n"),
                  color="#94a3b8", fontsize=8)
    ax.tick_params(colors="#475569", labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e293b")

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                val + 0.02, f"{val:.2f}",
                ha="center", color="#e2e8f0",
                fontsize=8)

plt.tight_layout()
plt.savefig("paper_figures/hqm_components.png",
            dpi=150, facecolor="#0f172a")
plt.show()
print("✓ Saved paper_figures/hqm_components.png")

In [ ]:
# Cell 6: Counterfactual Analysis
fig, ax = plt.subplots(figsize=(9, 4),
                        facecolor="#0f172a")
ax.set_facecolor("#020617")

ax.errorbar(cf["offset_s"], cf["hqm_mean"],
            yerr=cf["hqm_std"],
            color="#3b82f6", linewidth=2,
            marker="o", markersize=5,
            capsize=4, ecolor="#475569",
            label="Mean HQM")

ax.axhline(y=0.60, color="#ef4444",
            linestyle="--", linewidth=1,
            label="Greedy baseline")
ax.axvline(x=0.0, color="#475569",
            linestyle=":", linewidth=1,
            label="Actual commit time")

# Shade premature region
ax.axvspan(cf["offset_s"].min(), 0,
            alpha=0.08, color="#ef4444",
            label="Premature zone")

ax.set_xlabel("Commit Timing Offset (s)",
               color="#64748b")
ax.set_ylabel("Mean HQM", color="#64748b")
ax.set_title("Counterfactual Commitment Timing",
              color="#e2e8f0", fontsize=11)
ax.tick_params(colors="#475569")
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.legend(fontsize=8, facecolor="#0f172a",
           labelcolor="#94a3b8",
           edgecolor="#1e293b")

plt.tight_layout()
plt.savefig("paper_figures/counterfactual_notebook.png",
            dpi=150, facecolor="#0f172a")
plt.show()
print("✓ Saved paper_figures/counterfactual_notebook.png")

In [ ]:
# Cell 7: Parameter Sweep Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                          facecolor="#0f172a")
fig.suptitle("1000-Configuration Parameter Sweep",
              color="#e2e8f0", fontsize=11)

# Left: HQM distribution
ax = axes[0]
ax.set_facecolor("#020617")
ax.hist(sweep["mean_hqm"], bins=40,
         color="#3b82f6", alpha=0.8,
         edgecolor="#020617")
ax.axvline(x=0.60, color="#ef4444",
            linestyle="--", linewidth=1.5,
            label="Greedy baseline")
ax.axvline(x=sweep["mean_hqm"].mean(),
            color="#22c55e", linestyle="-",
            linewidth=1.5,
            label=f"Mean={sweep['mean_hqm'].mean():.3f}")
ax.set_xlabel("Mean HQM", color="#64748b")
ax.set_ylabel("Count", color="#64748b")
ax.set_title("HQM Distribution",
              color="#94a3b8", fontsize=9)
ax.tick_params(colors="#475569")
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.legend(fontsize=8, facecolor="#0f172a",
           labelcolor="#94a3b8",
           edgecolor="#1e293b")

# Right: Win rate text summary
ax2 = axes[1]
ax2.set_facecolor("#020617")
ax2.axis("off")

wins     = sweep["beats_greedy"].sum()
total    = len(sweep)
win_rate = 100 * wins / total

summary_text = (
    f"Configurations tested:   {total}\n\n"
    f"Beat greedy (HQM>0.60):  {wins}\n\n"
    f"Win rate:                {win_rate:.1f}%\n\n"
    f"Min HQM observed:        "
    f"{sweep['mean_hqm'].min():.4f}\n\n"
    f"Max HQM observed:        "
    f"{sweep['mean_hqm'].max():.4f}\n\n"
    f"Mean HQM:                "
    f"{sweep['mean_hqm'].mean():.4f}\n\n"
    f"Std HQM:                 "
    f"{sweep['mean_hqm'].std():.4f}"
)

ax2.text(0.1, 0.9, summary_text,
          transform=ax2.transAxes,
          color="#e2e8f0", fontsize=10,
          verticalalignment="top",
          fontfamily="monospace")
ax2.set_title("Robustness Summary",
               color="#94a3b8", fontsize=9)

plt.tight_layout()
plt.savefig("paper_figures/param_sweep_dist.png",
            dpi=150, facecolor="#0f172a")
plt.show()
print(f"✓ Win rate: {win_rate:.1f}%")
print("✓ Saved paper_figures/param_sweep_dist.png")

In [ ]:
# Cell 8: Sensitivity Analysis
params   = ["tau_l", "rho_c", "sigma_s"]
scenario_colors = {
    "pedestrian_curb":       "#3b82f6",
    "merge_hesitation":      "#22c55e",
    "occluded_intersection": "#f97316",
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4),
                          facecolor="#0f172a")
fig.suptitle("Parameter Sensitivity Analysis",
              color="#e2e8f0", fontsize=11)

baseline_hqm = sens[sens.param == "baseline"]\
               .groupby("scenario")["hqm_mean"].mean()

for col, param in enumerate(params):
    ax = axes[col]
    ax.set_facecolor("#020617")

    param_df  = sens[sens.param == param]
    scenarios = param_df["scenario"].unique()

    for scenario in scenarios:
        sdf  = param_df[param_df.scenario == scenario]
        pcts = sorted(sdf["perturbation"].unique())
        means = [sdf[sdf.perturbation==p]
                 ["hqm_mean"].values[0] for p in pcts]
        stds  = [sdf[sdf.perturbation==p]
                 ["hqm_std"].values[0]  for p in pcts]

        base = baseline_hqm.get(scenario, 0.685)
        pcts_full  = [-20, -10, 0, 10, 20]
        means_full = means[:2] + [base] + means[2:]
        stds_full  = stds[:2]  + [0.0]  + stds[2:]

        ax.plot(pcts_full, means_full,
                color=scenario_colors.get(
                    scenario, "#818cf8"),
                linewidth=2, marker="o",
                markersize=4,
                label=scenario.replace("_", "\n"))
        ax.fill_between(
            pcts_full,
            [m-s for m,s in zip(means_full,stds_full)],
            [m+s for m,s in zip(means_full,stds_full)],
            alpha=0.1,
            color=scenario_colors.get(
                scenario, "#818cf8"))

    ax.axhline(y=0.60, color="#ef4444",
                linestyle="--", linewidth=1)
    ax.axvline(x=0, color="#475569",
                linestyle=":", linewidth=1)
    ax.set_xlabel(f"{param} perturbation (%)",
                   color="#64748b", fontsize=8)
    ax.set_ylabel("Mean HQM", color="#64748b",
                   fontsize=8)
    ax.set_title(f"Sensitivity: {param}",
                  color="#94a3b8", fontsize=9)
    ax.set_xticks([-20, -10, 0, 10, 20])
    ax.tick_params(colors="#475569", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e293b")
    if col == 0:
        ax.legend(fontsize=6, facecolor="#0f172a",
                   labelcolor="#94a3b8",
                   edgecolor="#1e293b")

plt.tight_layout()
plt.savefig("paper_figures/sensitivity_notebook.png",
            dpi=150, facecolor="#0f172a")
plt.show()
print("✓ Saved paper_figures/sensitivity_notebook.png")

In [ ]:
# Cell 9: CARLA Results (if available)
if carla is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                              facecolor="#0f172a")
    fig.suptitle("CARLA Phase 2 Validation",
                  color="#e2e8f0", fontsize=11)

    # By scenario
    ax = axes[0]
    ax.set_facecolor("#020617")
    scenario_means = carla.groupby(
        "scenario")["hqm"].mean()
    scenario_stds  = carla.groupby(
        "scenario")["hqm"].std()

    colors = ["#3b82f6", "#22c55e", "#f97316"]
    bars = ax.bar(scenario_means.index,
                   scenario_means.values,
                   yerr=scenario_stds.values,
                   color=colors, alpha=0.85,
                   capsize=4, ecolor="#475569",
                   edgecolor="#020617")
    ax.axhline(y=0.60, color="#ef4444",
                linestyle="--", linewidth=1)
    ax.set_xticklabels(
        [s.replace("_", "\n")
         for s in scenario_means.index],
        fontsize=8, color="#94a3b8")
    ax.set_ylabel("Mean HQM", color="#64748b")
    ax.set_title("HQM by Scenario",
                  color="#94a3b8", fontsize=9)
    ax.tick_params(colors="#475569")
    for spine in ax.spines.values():
        spine.set_edgecolor("#1e293b")

    # By weather
    ax2 = axes[1]
    ax2.set_facecolor("#020617")
    weather_means = carla.groupby(
        "weather")["hqm"].mean()
    weather_stds  = carla.groupby(
        "weather")["hqm"].std()

    colors2 = ["#f97316", "#818cf8",
               "#22c55e", "#3b82f6"]
    ax2.bar(weather_means.index,
             weather_means.values,
             yerr=weather_stds.values,
             color=colors2, alpha=0.85,
             capsize=4, ecolor="#475569",
             edgecolor="#020617")
    ax2.axhline(y=0.60, color="#ef4444",
                 linestyle="--", linewidth=1)
    ax2.set_xticklabels(
        weather_means.index,
        fontsize=8, color="#94a3b8")
    ax2.set_ylabel("Mean HQM", color="#64748b")
    ax2.set_title("HQM by Weather",
                   color="#94a3b8", fontsize=9)
    ax2.tick_params(colors="#475569")
    for spine in ax2.spines.values():
        spine.set_edgecolor("#1e293b")

    plt.tight_layout()
    plt.savefig(
        "paper_figures/carla_notebook.png",
        dpi=150, facecolor="#0f172a")
    plt.show()
    print("Saved paper_figures/carla_notebook.png")
else:
    print("  Skipped : no CARLA results file found")

In [ ]:
# Cell 10: Summary Stats Table
print("\n" + "="*65)
print("  COMPLETE RESULTS SUMMARY")
print("="*65)

print("\n  Six-Policy HQM:")
print(summary[["scenario","policy",
               "hqm_mean","hqm_std",
               "S_mean","B_mean"]]\
      .to_string(index=False))

print(f"\n  Parameter Sweep:")
print(f"    Configs tested:  {len(sweep)}")
print(f"    Win rate:        "
      f"{100*sweep['beats_greedy'].mean():.1f}%")
print(f"    HQM range:       "
      f"{sweep['mean_hqm'].min():.4f} - "
      f"{sweep['mean_hqm'].max():.4f}")

print(f"\n  Counterfactual:")
worst = cf.loc[cf["hqm_mean"].idxmin()]
best  = cf.loc[cf["hqm_mean"].idxmax()]
print(f"    Worst offset:    "
      f"{worst['offset_s']}s "
      f"(HQM={worst['hqm_mean']:.4f})")
print(f"    Best offset:     "
      f"{best['offset_s']}s "
      f"(HQM={best['hqm_mean']:.4f})")

print("\n Analysis complete.")